In [ ]:
#!/usr/bin/env python3
"""
Fully Integrated AI Dashcam System
- Receives UDP streams from Raspberry Pi (front, rear, sensors)
- Buffers 20 seconds of video
- Runs collision detection (MobileNetV3-LSTM) on every frame
- On collision: saves 40s clip (20s pre + 10s during + 10s post)
- Runs all models on saved clip:
    - Lane departure detection (Hough classical)
    - Parking/obstacle detection (YOLO + depth)
    - Number plate extraction (user's perfect module)
    - Traffic light & traffic sign classification
    - DMS (if cabin cam available)
- Outputs fault report (ego/other fault percentages with confidence)
"""

In [6]:
import cv2
import numpy as np
import pickle
import socket
import time
import threading
import json
import os
import math
from collections import deque
from ultralytics import YOLO
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_large
from albumentations import Compose, Resize, Normalize
from albumentations.pytorch import ToTensorV2

In [7]:
MODEL_DIR = "models"
COLLISION_MODEL_PATH = os.path.join(MODEL_DIR, "MobileNetv3_collisionmodel.pth")
OBJECT_MODEL_PATH = os.path.join(MODEL_DIR, "object_detection.pt")
TRAFFIC_SIGNAL_PATH = os.path.join(MODEL_DIR, "traffic_signal.pt")
TRAFFIC_SIGN_PATH = os.path.join(MODEL_DIR, "TrafficSign.pt")
DMS_DROWSY_PATH = os.path.join(MODEL_DIR, "dms.pt")
DMS_OBJECTS_PATH = os.path.join(MODEL_DIR, "dms2.pt")

In [8]:
# Video settings
FRONT_WIDTH, FRONT_HEIGHT = 640, 480
REAR_WIDTH, REAR_HEIGHT = 640, 480
FPS = 30
BUFFER_SECONDS = 20
FRAMES_TO_KEEP = BUFFER_SECONDS * FPS

# Collision thresholds (from your notebook)
COLLISION_HIGH_THRESH = 0.33
COLLISION_LOW_THRESH = 0.25
COLLISION_MIN_CONFIRM = 8

# Fault weights
WEIGHTS = {
    "lane_departure_ego": 25,
    "following_too_close_ego": 40,
    "red_light_ego": 50,
    "speeding_ego": 30,
    "drowsy_ego": 35,
    "phone_use_ego": 35,
    "parking_obstacle_ego": 20,
    "other_cut_in": 40,
    "other_red_light": 50,
}
# UDP settings
UDP_IP = "0.0.0.0"
UDP_PORT = 5005

In [9]:
# ======================= COLLISION MODEL =======================
class CollisionPredictor(nn.Module):
    def __init__(self, lstm_hidden=256):
        super().__init__()
        self.backbone = mobilenet_v3_large(weights=None)
        backbone_out = self.backbone.classifier[0].in_features
        self.backbone.classifier = nn.Identity()
        self.lstm = nn.LSTM(backbone_out, lstm_hidden, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        b, t, c, h, w = x.shape
        x = x.view(b * t, c, h, w)
        features = self.backbone(x).view(b, t, -1)
        lstm_out, _ = self.lstm(features)
        return self.classifier(lstm_out[:, -1, :]).squeeze(-1)

class CollisionDetector:
    def __init__(self, model_path, device='cuda'):
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        self.model = CollisionPredictor().to(self.device)
        state = torch.load(model_path, map_location=self.device)
        self.model.load_state_dict(state)
        self.model.eval()
        self.transform = Compose([
            Resize(224, 224),
            Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
            ToTensorV2(),
        ])
        self.frame_window = deque(maxlen=16)
    def process_frame(self, frame_bgr):
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        self.frame_window.append(frame_rgb)
        if len(self.frame_window) == 16:
            tensor = torch.stack([
                self.transform(image=img)["image"] for img in self.frame_window
            ]).unsqueeze(0).to(self.device)
            with torch.no_grad():
                prob = torch.sigmoid(self.model(tensor)).item()
            return prob
        return 0.0

In [10]:
# ======================= LANE DETECTION (Classical Hough) =======================
def detect_lane_departure(frame):
    """Return True if lane departure (only one lane line detected)"""
    height, width = frame.shape[:2]
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    roi = np.array([[(0, height), (width//2, height//2), (width, height)]], dtype=np.int32)
    mask = np.zeros_like(edges)
    cv2.fillPoly(mask, roi, 255)
    cropped = cv2.bitwise_and(edges, mask)
    lines = cv2.HoughLinesP(cropped, rho=6, theta=np.pi/180, threshold=120,
                            minLineLength=30, maxLineGap=100)
    if lines is None:
        return False
    left_slopes = []
    right_slopes = []
    for line in lines:
        for x1,y1,x2,y2 in line:
            if x2 - x1 == 0:
                continue
            slope = (y2 - y1) / (x2 - x1)
            if abs(slope) < 0.5:
                continue
            if slope < 0:
                left_slopes.append(slope)
            else:
                right_slopes.append(slope)
    # Departure if one side is missing
    return (len(left_slopes) == 0 or len(right_slopes) == 0)

In [11]:
# ======================= PARKING / OBSTACLE DETECTION =======================
# We'll use YOLO + depth from MiDaS (if available), else fallback to YOLO + bottom y
# Load MiDaS only once (optional)
midas_model = None
midas_transform = None
def init_midas():
    global midas_model, midas_transform
    if midas_model is None:
        midas_model = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
        midas_model.eval()
        midas_transform = torch.hub.load("intel-isl/MiDaS", "transforms").small_transform
    return midas_model, midas_transform

def get_parking_risk(frame, yolo_model, use_depth=False):
    """
    Returns (risk_level, distance_meters)
    risk_level: "STOP", "CAUTION", "GO_SLOW", "CLEAR"
    """
    h, w = frame.shape[:2]
    # Region of interest (trapezoid)
    roi_pts = np.array([
        [int(w*0.30), int(h*0.60)],
        [int(w*0.70), int(h*0.60)],
        [int(w*0.95), h],
        [int(w*0.05), h]
    ], np.int32)
    # Use depth if requested and available
    if use_depth:
        midas, transform = init_midas()
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        input_batch = transform(img_rgb).unsqueeze(0)
        with torch.no_grad():
            depth = midas(input_batch)
            depth = torch.nn.functional.interpolate(
                depth.unsqueeze(1), size=(h,w), mode='bicubic'
            ).squeeze().cpu().numpy()
        # Normalize depth (0=far, 1=near)
        depth = cv2.normalize(depth, None, 0, 1, cv2.NORM_MINMAX)
        mask = np.zeros_like(depth)
        cv2.fillPoly(mask, [roi_pts], 1)
        roi_depth = depth * mask
        close_ratio = np.sum(roi_depth > 0.7) / (np.sum(mask)+1e-6)
        if close_ratio > 0.6:
            return "STOP", 0.5
        elif close_ratio > 0.3:
            return "CAUTION", 1.2
        else:
            return "CLEAR", 5.0
    else:
        # Simple YOLO-based distance using bottom y coordinate
        results = yolo_model(frame, verbose=False)[0]
        max_bottom = 0
        for box in results.boxes:
            cls = int(box.cls[0])
            if cls in [0,2,3,5,7]:  # person, car, bus, truck
                x1,y1,x2,y2 = map(int, box.xyxy[0])
                cx, cy = (x1+x2)//2, y2
                if cv2.pointPolygonTest(roi_pts, (cx, cy), False) >= 0:
                    max_bottom = max(max_bottom, cy)
        if max_bottom > h * 0.85:
            return "STOP", 0.5
        elif max_bottom > h * 0.75:
            return "CAUTION", 1.2
        elif max_bottom > h * 0.65:
            return "GO_SLOW", 2.5
        else:
            return "CLEAR", 10.0


In [12]:
class TrafficLightDetector:
    def __init__(self, model_path, device='cpu'):
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        self.model = torch.load(model_path, map_location=self.device)
        self.model.eval()
    def get_state(self, frame):
        h,w = frame.shape[:2]
        top = frame[0:int(h*0.4), :]
        if top.size == 0:
            return -1  # unknown
        inp = cv2.resize(top, (224,224))
        inp_t = torch.from_numpy(inp).float().permute(2,0,1).unsqueeze(0).to(self.device)
        with torch.no_grad():
            out = self.model(inp_t)
        return torch.argmax(out, dim=1).item()  # 0=red,1=yellow,2=green

class TrafficSignDetector:
    def __init__(self, model_path, device='cpu'):
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        self.model = torch.load(model_path, map_location=self.device)
        self.model.eval()
    def detect_sign(self, frame):
        # Simple sliding window on top region, but for demo resize whole image
        inp = cv2.resize(frame, (224,224))
        inp_t = torch.from_numpy(inp).float().permute(2,0,1).unsqueeze(0).to(self.device)
        with torch.no_grad():
            out = self.model(inp_t)
        return torch.argmax(out, dim=1).item()  # class id

In [13]:
# ======================= NUMBER PLATE (your perfect module) =======================
# Assuming your numberplate.py has a function detect_plate(frame) returning string
try:
    from numberplate import detect_plate as numberplate_detect
    NUMBERPLATE_AVAILABLE = True
except ImportError:
    NUMBERPLATE_AVAILABLE = False
    def numberplate_detect(frame):
        return None



In [14]:
# ======================= DMS MODELS (Drowsiness + Objects) =======================
class DMSDrowsyDetector:
    def __init__(self, model_path, device='cpu'):
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        self.model = torch.load(model_path, map_location=self.device)
        self.model.eval()
    def detect(self, face_crop):
        # You need to implement preprocessing (e.g., resize to 224x224)
        inp = cv2.resize(face_crop, (224,224))
        inp_t = torch.from_numpy(inp).float().permute(2,0,1).unsqueeze(0).to(self.device)
        with torch.no_grad():
            out = self.model(inp_t)
        # Assume output is a probability for drowsy (0-1)
        return float(out[0,0])

class DMSObjectsDetector:
    def __init__(self, model_path, device='cpu'):
        self.device = torch.device(device if torch.cuda.is_available() else 'cpu')
        self.model = torch.load(model_path, map_location=self.device)
        self.model.eval()
    def detect(self, face_crop):
        """Returns (cigarette_prob, bottle_prob, mobile_prob)"""
        inp = cv2.resize(face_crop, (224,224))
        inp_t = torch.from_numpy(inp).float().permute(2,0,1).unsqueeze(0).to(self.device)
        with torch.no_grad():
            out = self.model(inp_t)  # shape (1,3) for three classes
        probs = torch.sigmoid(out).squeeze().cpu().numpy()
        return probs[0], probs[1], probs[2]   # cig, bottle, mobile

In [15]:
def save_40sec_clip(front_buffer, rear_buffer, sensor_buffer, collision_time=None):
    """
    Saves 20s pre, 10s during, 10s post from buffers.
    Returns dict with file paths.
    """
    # For simplicity, we save the entire buffer as pre and then simulate.
    # In real implementation, you would record additional frames after trigger.
    os.makedirs("clips", exist_ok=True)
    timestamp = int(time.time())
    front_path = f"clips/event_front_{timestamp}.mp4"
    rear_path = f"clips/event_rear_{timestamp}.mp4"
    sensor_path = f"clips/event_sensor_{timestamp}.json"
    # Save front video (all frames in buffer)
    out = cv2.VideoWriter(front_path, cv2.VideoWriter_fourcc(*'mp4v'), FPS,
                          (FRONT_WIDTH, FRONT_HEIGHT))
    for _, frame in front_buffer:
        out.write(frame)
    out.release()
    # Save rear video
    out = cv2.VideoWriter(rear_path, cv2.VideoWriter_fourcc(*'mp4v'), FPS,
                          (REAR_WIDTH, REAR_HEIGHT))
    for _, frame in rear_buffer:
        out.write(frame)
    out.release()
    # Save sensor log
    with open(sensor_path, 'w') as f:
        json.dump([(ts, data) for ts, data in sensor_buffer], f)
    return {"front": front_path, "rear": rear_path, "sensor": sensor_path}

In [16]:
def compute_fault_report(evidence, weights):
    ego_score = 0
    other_score = 0
    if evidence.get("lane_departures", 0) > 2:
        ego_score += weights["lane_departure_ego"]
    if evidence.get("close_following", False):
        ego_score += weights["following_too_close_ego"]
    if evidence.get("red_light_ego", False):
        ego_score += weights["red_light_ego"]
    if evidence.get("speeding", False):
        ego_score += weights["speeding_ego"]
    if evidence.get("drowsy", False):
        ego_score += weights["drowsy_ego"]
    if evidence.get("cigarette", False):
        ego_score += weights["cigarette_ego"]
    if evidence.get("bottle", False):
        ego_score += weights["bottle_ego"]
    if evidence.get("mobile", False):
        ego_score += weights["mobile_ego"]
    if evidence.get("parking_obstacle", False):
        ego_score += weights["parking_obstacle_ego"]
    if evidence.get("other_cut_in", False):
        other_score += weights["other_cut_in"]
    if evidence.get("other_red_light", False):
        other_score += weights["other_red_light"]
    total = ego_score + other_score
    if total == 0:
        ego_fault = 50
        other_fault = 50
        conf = 0.5
    else:
        ego_fault = (ego_score / total) * 100
        other_fault = 100 - ego_fault
        conf = min(0.9, 0.5 + total/200)
    return {
        "ego_fault_percentage": round(ego_fault, 1),
        "other_fault_percentage": round(other_fault, 1),
        "confidence": round(conf, 2),
        "evidence": evidence
    }


In [17]:
def analyze_collision_clip(front_video_path, rear_video_path, sensor_json_path, cabin_video_path=None):
    """If cabin_video_path is provided, run DMS models."""
    yolo_model = YOLO(OBJECT_MODEL_PATH)
    traffic_light = TrafficLightDetector(TRAFFIC_SIGNAL_PATH)
    traffic_sign = TrafficSignDetector(TRAFFIC_SIGN_PATH)
    # Load DMS models if cabin video is available
    dms_drowsy = None
    dms_objects = None
    if cabin_video_path and os.path.exists(DMS_DROWSY_PATH) and os.path.exists(DMS_OBJECTS_PATH):
        dms_drowsy = DMSDrowsyDetector(DMS_DROWSY_PATH)
        dms_objects = DMSObjectsDetector(DMS_OBJECTS_PATH)
        cap_cabin = cv2.VideoCapture(cabin_video_path)
    else:
        cap_cabin = None
    cap_front = cv2.VideoCapture(front_video_path)
    cap_rear = cv2.VideoCapture(rear_video_path)
    evidence = {
        "lane_departures": 0,
        "close_following": False,
        "red_light_ego": False,
        "speeding": False,
        "drowsy": False,
        "cigarette": False,
        "bottle": False,
        "mobile": False,
        "parking_obstacle": False,
        "other_cut_in": False,
        "other_red_light": False,
    }
    frame_count = 0
    while True:
        ret_f, front = cap_front.read()
        ret_r, rear = cap_rear.read()
        if not ret_f or not ret_r:
            break
        if cap_cabin:
            ret_c, cabin = cap_cabin.read()
            if not ret_c:
                cabin = None
        else:
            cabin = None
        if frame_count % 5 == 0:
            # Lane departure
            if detect_lane_departure(front):
                evidence["lane_departures"] += 1
            # Parking risk (rear)
            risk, dist = get_parking_risk(rear, yolo_model)
            if risk in ["STOP", "CAUTION"]:
                evidence["parking_obstacle"] = True
            # Following distance
            h_f = front.shape[0]
            results = yolo_model(front, verbose=False)[0]
            for box in results.boxes:
                cls = int(box.cls[0])
                if cls in [2,3,5,7]:
                    x1,y1,x2,y2 = map(int, box.xyxy[0])
                    if y2 > h_f * 0.85:
                        evidence["close_following"] = True
            # Traffic light
            light_state = traffic_light.get_state(front)
            if light_state == 0:
                evidence["red_light_ego"] = True
            # DMS on cabin frame (if available)
            if cabin is not None and dms_drowsy and dms_objects:
                # Assume you have a face detector or use the whole frame as driver area
                # For simplicity, use the whole frame as crop (you should implement face detection)
                drowsy_prob = dms_drowsy.detect(cabin)
                cig, bottle, mobile = dms_objects.detect(cabin)
                if drowsy_prob > 0.7:
                    evidence["drowsy"] = True
                if cig > 0.7:
                    evidence["cigarette"] = True
                if bottle > 0.7:
                    evidence["bottle"] = True
                if mobile > 0.7:
                    evidence["mobile"] = True
            # Number plate (rear)
            if NUMBERPLATE_AVAILABLE:
                plate = numberplate_detect(rear)
                if plate:
                    evidence["other_vehicle_detected"] = True
        frame_count += 1
        if frame_count > 300:
            break
    cap_front.release()
    cap_rear.release()
    if cap_cabin:
        cap_cabin.release()
    return compute_fault_report(evidence, WEIGHTS)


In [20]:
front_buffer = deque(maxlen=FRAMES_TO_KEEP)
rear_buffer = deque(maxlen=FRAMES_TO_KEEP)
sensor_buffer = deque(maxlen=FRAMES_TO_KEEP)
collision_detector = CollisionDetector(COLLISION_MODEL_PATH)

def udp_receiver():
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.bind((UDP_IP, UDP_PORT))
    while True:
        data, addr = sock.recvfrom(65536)
        packet = pickle.loads(data)
        cam = packet['cam']
        ts = packet['ts']
        content = packet['data']
        if cam == 'front':
            front_buffer.append((ts, content))
        elif cam == 'rear':
            rear_buffer.append((ts, content))
        elif cam == 'sensor':
            sensor_buffer.append((ts, content))

def on_collision():
    print("\n🚨 Collision detected! Saving 40-second clip...")
    paths = save_40sec_clip(front_buffer, rear_buffer, sensor_buffer)
    print("Analyzing clip for fault attribution...")
    # If you have a cabin camera video, pass its path here
    report = analyze_collision_clip(paths['front'], paths['rear'], paths['sensor'], cabin_video_path=None)
    print("\n========== FAULT REPORT ==========")
    print(f"Ego vehicle fault: {report['ego_fault_percentage']}%")
    print(f"Other vehicle fault: {report['other_fault_percentage']}%")
    print(f"Confidence: {report['confidence']}")
    print("Evidence collected:", report['evidence'])
    os.makedirs("logs", exist_ok=True)
    with open(f"logs/fault_{int(time.time())}.json", "w") as f:
        json.dump(report, f)

In [ ]:
def main():
    threading.Thread(target=udp_receiver, daemon=True).start()
    print("System ready. Press 'c' to simulate collision, 'q' to quit.")
    event_triggered = False
    last_trigger_time = 0
    while True:
        if front_buffer:
            ts, frame = front_buffer[-1]
            prob = collision_detector.process_frame(frame)
            if prob > COLLISION_HIGH_THRESH and not event_triggered and (time.time() - last_trigger_time) > 3:
                event_triggered = True
                on_collision()
                last_trigger_time = time.time()
                event_triggered = False
            cv2.putText(frame, f"Collision Prob: {prob:.2f}", (10,30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
            cv2.imshow("AI Dashcam", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):
            break
        elif key == ord('c'):
            on_collision()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Exception in thread Thread-4:
Traceback (most recent call last):
  File "C:\Users\soham\anaconda3\envs\FinalPipeline\lib\threading.py", line 980, in _bootstrap_inner
    self.run()
  File "C:\Users\soham\anaconda3\envs\FinalPipeline\lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\soham\anaconda3\envs\FinalPipeline\lib\threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\soham\AppData\Local\Temp\ipykernel_5936\569994665.py", line 8, in udp_receiver
OSError: [WinError 10048] Only one usage of each socket address (protocol/network address/port) is normally permitted


System ready. Press 'c' to simulate collision, 'q' to quit.
